In [ ]:
import sys
import torch
import torch.nn as nn
import torchvision
from tqdm import tqdm
from torchvision import transforms
from torch.utils.data import random_split, DataLoader
from PIL import Image

device = torch.device("cpu")
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.mps.is_available():
    device = torch.device("mps")

# 데이터 전처리: 이미지를 그대로 사용할 수 없고 텐서로 변경해야 함
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

resnet50_model = torchvision.models.resnet50(
    weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V1
)

resnet50_model.fc = nn.Identity()  # .fc > 자동으로 마지막 레이어에 덮어쓰는 방법
resnet50_model = resnet50_model.to(device)

fc_model = nn.Sequential(
    nn.Linear( 2048, 1024 ),
    nn.ReLU(),
    nn.Linear(1024, 1)
)

fc_static_dict = torch.load("fc_model_0.pth", weights_only=True )
fc_model.load_state_dict(fc_static_dict)
fc_model = fc_model.to(device)

model = nn.Sequential(
    resnet50_model,
    fc_model
)
model = model.to(device)
model.eval()

tire1 = Image.open("./images" ) # 내차 타이어 추가할 것
tire1_tensor = preprocess( tire1 )
tire1_tensor = tire1_tensor.unsqueeze(dim=0)
tire1_tensor = tire1_tensor.to(device)

with torch.no_grad():
    y_pred = torch.sigmoid( model(tire1_tensor) )
    print( y_pred )
    pass


OrderedDict([('0.weight', tensor([[-0.0155,  0.0189,  0.0217,  ...,  0.0181, -0.0012, -0.0121],
        [-0.0212, -0.0033,  0.0100,  ...,  0.0195, -0.0193, -0.0191],
        [-0.0179, -0.0048,  0.0042,  ..., -0.0129,  0.0030, -0.0137],
        ...,
        [ 0.0114,  0.0084,  0.0184,  ..., -0.0217,  0.0080,  0.0062],
        [-0.0084,  0.0216, -0.0226,  ..., -0.0003, -0.0118,  0.0049],
        [ 0.0051, -0.0178, -0.0092,  ..., -0.0235,  0.0061, -0.0217]])), ('0.bias', tensor([-0.0048,  0.0198, -0.0072,  ...,  0.0108,  0.0062, -0.0112])), ('2.weight', tensor([[-0.0165,  0.0260,  0.0244,  ...,  0.0261,  0.0124,  0.0155]])), ('2.bias', tensor([-0.0063]))])
